## Data Understanding and Preparation

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from IPython.display import display
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

### Data Loading

In [ ]:
# Load CRM Contacts Data
# Load the most recent CRM contacts
crm_contacts = pd.read_excel(
    f'/Users/meecee/Desktop/Github/Networking Recommendation System/data/raw/CRM Contacts Sept25.xlsx',
    sheet_name='person list', header=2)

print(f"Loaded {len(crm_contacts)} contacts from Sept 2025")

print('')
print('Sample rows of recent contact data')
display(crm_contacts.head(2))

# Load historical contacts for comparison
crm_contacts_23 = pd.read_excel(
    f'/Users/meecee/Desktop/Github/Networking Recommendation System/data/raw/CRM Contacts 04.05.23.xlsx',
    sheet_name='people list', header=2)

print(f"Loaded {len(crm_contacts_23)} contacts from May 2023")

print('')
print('\nSample rows of historical contact data')
display(crm_contacts_23.head(2))

In [ ]:
# Load organisation data
crm_orgs = pd.read_excel(f'/Users/meecee/Desktop/Github/Networking Recommendation System/data/raw/CRM Organisations Sept25.xlsx',
                         sheet_name='organization list', header=2)
print('\n')
print(f"Loaded {len(crm_orgs)} organisations from Sept 2025")
print('\nSample rows of organisation data')
display(crm_orgs.head(2))

In [ ]:
# Load Event Attendance Data
# Load Eventbrite historical data
event_history = pd.read_excel(f'/Users/meecee/Desktop/Github/Networking Recommendation System/data/raw/CyNam Event History Attendee List.xlsx',
                              sheet_name='Sheet 1', header=2)

print('\n')
print(f"Loaded {len(event_history)} event attendees from Eventbrite")
print('\nSample rows of historical event data')
display(event_history.head(2))

In [ ]:
# Load Luma event data (recent events)
luma_file = pd.ExcelFile(
    f'/Users/meecee/Desktop/Github/Networking Recommendation System/data/raw/Luma Event History Attendee List.xlsx')
luma_events = []

for sheet in luma_file.sheet_names:
    if sheet != 'Tabelle1':  # Skip empty sheet
        df = pd.read_excel(luma_file, sheet_name=sheet)
        if len(df) > 0:
            df['event_name'] = sheet
            luma_events.append(df)

events_luma = pd.concat(luma_events, ignore_index=True)
print('')
print(f"Loaded {len(events_luma)} event attendees from Luma ({len(luma_file.sheet_names)-1} events)")
display(events_luma.head(2))

In [ ]:
# Let's see how many names match between the two CRM files
names_25 = set(crm_contacts['Person - Name'].unique())
names_23 = set(crm_contacts_23['CRM Contact Name'].unique())
common_names = names_25.intersection(names_23)
print(f"\nUnique names in 2025 CRM: {len(names_25)}")
print(f"Unique names in 2023 CRM: {len(names_23)}")
print(f"Common names: {len(common_names)}")

In [ ]:
# Mailchimp files inspection
mailchimp_sub = pd.read_excel('/Users/meecee/Desktop/Github/Networking Recommendation System/data/raw/Cynam Mailchimp Subscribed contact list.xlsx',
                              sheet_name='Cynam Mailchimp Subscribed cont', header=2)
mailchimp_unsub = pd.read_excel('/Users/meecee/Desktop/Github/Networking Recommendation System/data/raw/CyNam Mailchimp unsubscribed contact list.xlsx',
                                sheet_name='CyNam Mailchimp unsubscribed co', header=2)
mailchimp_non = pd.read_excel('/Users/meecee/Desktop/Github/Networking Recommendation System/data/raw/Cynam Mailchimp Non-subscribed contact list.xlsx',
                              sheet_name='nonsubscribed_email_audience_ex', header=2)

print('\n')
print(f"Loaded {len(mailchimp_sub)} subscribed mailchimp contacts")
display(mailchimp_sub[['Email Address', 'First Name',
        'Last Name', 'Company', 'Job Title', 'Sector']].head(2))

print('')
print(f"Loaded {len(mailchimp_unsub)} unsubscribed mailchimp contacts")
display(mailchimp_unsub[['Email Address', 'First Name',
        'Last Name', 'Company', 'Job Title', 'Sector']].head(2))

print('')
print(f"Loaded {len(mailchimp_non)} non-subscribed mailchimp contacts")
display(mailchimp_non[['Email Address', 'First Name',
        'Last Name', 'Company', 'Job Title', 'Sector']].head(2))

### Master Member Profile Creation

In [ ]:
# Gather all (Email, First Name, Last Name) combinations from all sources
def get_emails(df, email_col, fname_col, lname_col, source_name):
    # Standardize column names
    temp = df[[email_col, fname_col, lname_col]].copy()
    temp.columns = ['email', 'first_name', 'last_name']
    temp['source'] = source_name
    return temp

In [ ]:
# List of all files with email info
email_sources = [
    (mailchimp_sub, 'Email Address', 'First Name', 'Last Name', 'Mailchimp_Sub'),
    (mailchimp_unsub, 'Email Address', 'First Name', 'Last Name', 'Mailchimp_Unsub'),
    (mailchimp_non, 'Email Address', 'First Name', 'Last Name', 'Mailchimp_Non'),
    (event_history, 'Attendee email', 'Attendee first name',
     'Attendee last name', 'Eventbrite'),
    (crm_contacts_23, 'Email', 'CRM Contact Name',
     'CRM contact surname', 'CRM_2023'),
    (events_luma, 'email', 'first_name', 'last_name', 'Luma'),
]

# Gather all emails from the sources
all_emails = []
for df, e, f, l, s in email_sources:
    all_emails.append(get_emails(df, e, f, l, s))

# Combine all emails into a master DataFrame
master_members = pd.concat(all_emails, ignore_index=True)

# Clean emails
master_members['email'] = master_members['email'].str.lower().str.strip()
# Drop rows with no email
master_members = master_members.dropna(subset=['email'])

# Keep the first occurrence of each email (ideally we'd prioritize some sources, but this is a start)
master_members = master_members.sort_values(
    'source', ascending=False).drop_duplicates('email')

print(f"Total unique members identified by email: {len(master_members)}")
display(master_members.head())

In [ ]:
# Build a lookup for additional attributes from all sources
# Note: master_members has email, first_name, last_name
master_members['full_name'] = (master_members['first_name'].fillna(
    '') .astype(str) + ' ' + master_members['last_name'].fillna('').astype(str)).str.strip().str.lower()

# Gather attributes from sources that have Email
# Mailchimp sources:
mc_attrs = pd.concat([
    mailchimp_sub[['Email Address', 'Company', 'Job Title', 'Sector']],
    mailchimp_unsub[['Email Address', 'Company', 'Job Title', 'Sector']],
    mailchimp_non[['Email Address', 'Company', 'Job Title', 'Sector']]
])
mc_attrs['Email Address'] = mc_attrs['Email Address'].str.lower().str.strip()
mc_attrs = mc_attrs.drop_duplicates('Email Address')

# Merge mc_attrs into master_members
master_members = master_members.merge(
    mc_attrs, left_on='email', right_on='Email Address', how='left').drop(columns=['Email Address'])
# CRM 2025 doesn't have email, but has Name.
# Let's clean the CRM 2025 names
crm_contacts['full_name_clean'] = crm_contacts['Person - Name'].str.strip().str.lower()
# Create a lookup for CRM 2025
crm_25_lookup = crm_contacts[['full_name_clean', 'Person - Job Title',
                              'Organization - Name']].drop_duplicates('full_name_clean')

# Merge CRM 2025 info based on full_name
master_members = master_members.merge(crm_25_lookup, left_on='full_name',
                                      right_on='full_name_clean', how='left').drop(columns=['full_name_clean'])

# Consolidate columns (e.g. Job Title from Mailchimp vs CRM)
master_members['job_title'] = master_members['Person - Job Title'].fillna(
    master_members['Job Title'])
master_members['organization'] = master_members['Organization - Name'].fillna(
    master_members['Company'])
master_members['sector'] = master_members['Sector']

# Merge Organization sector/size from crm_orgs
crm_orgs_clean = crm_orgs[['Organization - Name', 'Organization - Sector',
                           'Organization - Company Size']].drop_duplicates('Organization - Name')
master_members = master_members.merge(
    crm_orgs_clean, left_on='organization', right_on='Organization - Name', how='left')

# Final consolidation of sector
master_members['sector'] = master_members['sector'].fillna(
    master_members['Organization - Sector'])
master_members['company_size'] = master_members['Organization - Company Size']

# Keep useful columns
final_members = master_members[['email', 'first_name', 'last_name',
                                'full_name', 'job_title', 'organization', 'sector', 'company_size']].copy()

print(f"Final member list size: {len(final_members)}")
display(final_members.head())
display(final_members.info())

### Event Attendance Profile Creation

In [ ]:
# Eventbrite events
eventbrite_history = event_history[['Attendee email', 'Event name']].dropna()
eventbrite_history.columns = ['email', 'event_name']
eventbrite_history['email'] = eventbrite_history['email'].str.lower(
).str.strip()

# Luma events
events_luma_emailname = events_luma[['email']].dropna()
events_luma_emailname['event_name'] = events_luma['event_name']
events_luma_emailname['email'] = events_luma_emailname['email'].str.lower(
).str.strip()

In [ ]:
# CRM 2025 events (linked by name to email)
# First, create name then email mapping
name_to_email = final_members[['full_name', 'email']].dropna().drop_duplicates(
    'full_name').set_index('full_name')['email'].to_dict()

crm_events_list = []
for idx, row in crm_contacts.iterrows():
    name = str(row['Person - Name']).strip().lower()
    events_str = row['Person - Events Attended/Registered']
    if pd.notna(events_str) and name in name_to_email:
        email = name_to_email[name]
        # Split events by comma
        events = [e.strip() for e in str(events_str).split(',')]
        for e in events:
            if e:
                crm_events_list.append({'email': email, 'event_name': e})

crm_history = pd.DataFrame(crm_events_list)

In [ ]:
# Combine all
all_event_attendance = pd.concat(
    [eventbrite_history, events_luma_emailname, crm_history]).drop_duplicates()

print(f"Total event attendance records: {len(all_event_attendance)}")
display(all_event_attendance.head())

In [ ]:
# Check top events
print("\nTop Events:")
display(all_event_attendance['event_name'].value_counts().head(10))

In [ ]:
# Save the cleaned member list and event attendance to CSV
final_members.to_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/cleaned_members.csv', index=False)
all_event_attendance.to_csv(
    '/Users/meecee/Desktop/Github/Networking Recommendation System/data/processed/event_attendance.csv', index=False)
